# **Proyecto Etapa 1. Selección de dataset para su procesamiento con PySparks**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|-----------|
| Ana Bonavides Aguilar | A01423281 |
|  | A01797775 |
|  | A01174130 |
|  | A01139580 |

# GTEx

## Dataset Seleccionado
El dataset que seleccionamos es el de Transcripción de Expresión del Genoma Por Millón (TPM) del RNA SeQCv2, que viene del proyecto de Expresión Genoyipo-Tejido (The Genotype-Tissue Expression (GTEx) project GTEx). Este archivo se encuentra en el GTEx portal, que, como ellos lo describen, es un "Recurso público integral para investigadores que estudian la expresión y regulación génica específica de tejidos y células en individuos, a lo largo del desarrollo y en diferentes especies, con datos de 3 proyectos de los NIH." (GTEx portal, 2026)

### Información importante
#### Nombre del Dataset
GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz	Gene Expression Transcripts Per Million (TPM) from RNASeQCv2.4.2.

#### Descripción corta
"The GTEx Analysis V10 release is the most complete analyzed dataset for Adult GTEx. It includes a new data type, smallRNA-seq." (GTEx portal, 2026)

#### Dónde encontrarlo
[Link al portal](https://gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression): https://gtexportal.org/home/downloads/adult-gtex/bulk_tissue_expression
[Link a descarga](https://storage.googleapis.com/adult-gtex/bulk-gex/v10/rna-seq/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz): https://storage.googleapis.com/adult-gtex/bulk-gex/v10/rna-seq/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_tpm.gct.gz


## Objetivo General

Escogimos esta base de datos ya que hay muchas opciones de posibles proyectos. Uno de los que más nos interesa, es la intersección de la Genómica y los viajes espaciales. Lo que proponemos es analizar los patrones de expresión génica en múltiples tipos de tejido humano para identificar marcadores biológicos relevantes para el monitoreo de la salud de astronautas, estableciendo una línea base genómica terrestre que la medicina espacial utiliza para detectar desviaciones causadas por el vuelo espacial. Como lo mencionan Shirah, B. et, al, "(...) la investigación espacial ha proporcionado un gran número de descubrimientos e invenciones en el campo de la medicina." (Shirah, B. et, al, 2023)

### Contexto
Con la misión Artemis II, y lo que viene, la medicina espacial estudia activamente cómo el vuelo espacial altera el cuerpo humano a nivel molecular (WEM, 2026). La microgravedad, la radiación cósmica y el estrés de misión modifican la **expresión génica** - qué tan activamente funcionan los genes - sin cambiar el ADN subyacente. Para detectar estas desviaciones, los investigadores necesitan una línea base terrestre normal.

El dataset GTEx proporciona exactamente eso: un mapa completo de la expresión génica humana en 54 tipos de tejido de casi 1,000 donantes sanos, financiado por el NIH Common Fund y mantenido por el Broad Institute del MIT y Harvard (GTEx, 2026)

# Bibliografía
- GTEx Portal (2026). _Download Open Access Datasets_
- Shirah, B., Bukhari, H., Pandya, S., & Ezmeirlly, H. A. (2023). _Benefits of Space Medicine Research for Healthcare on Earth._ Cureus, 15(5), e39174. https://doi.org/10.7759/cureus.39174
- World Extreme Medicine (2026). _SPACE MEDICINE COURSE_ . https://worldextrememedicine.com/extreme-medicine-courses/space-medicine-course/

## Dataset

| Campo | Detalle |
|-------|---------|
| **Nombre** | GTEx Analysis V10 - Gene Expression TPM |
| **Origen** | NIH / Broad Institute - Genotype-Tissue Expression Project |
| **Liga** | https://gtexportal.org/home/downloads/adult-gtex |
| **Archivo** | `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct` |
| **Tamaño** | ~6.8 GB (sin comprimir) - supera el requisito mínimo de 1 GB |
| **Formato** | GCT 1.2 (TSV con 2 filas de metadatos al inicio) |
| **Contenido** | ~59,000 genes × ~19,600 muestras de tejido humano |
| **Acceso** | Abierto - sin login ni IRB requerido |

El dataset contiene valores de **TPM (Transcripts Per Million)**: una medida de expresión génica normalizada por longitud del gen y profundidad de secuenciación, comparable entre muestras.

In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType
from functools import reduce

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("GTEx_TC5057") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [ ]:
import sys, os

# add src/ to path - notebook is at src/notebooks/, GlobalVariables.py is at src/
sys.path.insert(0, os.path.abspath('..'))
from GlobalVariables import FILE_PATH, OUTPUT_DIR, N_GENES, N_SAMPLES, SAMPLE_STEP, THRESHOLD, SPARK_MEMORY, APP_NAME, RANDOM_SEED

# set seeds so any random ops are reproducible across runs
import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'FILE_PATH:   {FILE_PATH}')
print(f'OUTPUT_DIR:  {OUTPUT_DIR}')
print(f'RANDOM_SEED: {RANDOM_SEED}')


## 1. Estructura del Dataset

El formato GCT 1.2 tiene 2 líneas de metadatos al inicio del archivo antes del encabezado real:
- **Línea 1:** `#1.2` - versión del formato
- **Línea 2:** `59033\t19616` - dimensiones del dataset
- **Línea 3:** encabezado real con `Name`, `Description` y los IDs de las ~19,600 muestras

Esto hace que `spark.read.csv(comment="#")` no funcione directamente - saltaría la línea 1 pero trataría la línea 2 (las dimensiones) como encabezado. La solución es extraer los nombres de columnas con pandas (solo lee 3 líneas) y cargar los datos con PySpark filtrando por contenido.

In [ ]:
# grab column names with pandas - way faster than spark for reading just 3 lines
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=3)
all_col_names = peek.columns.tolist()

n_samples = len(all_col_names) - 2  # minus Name and Description

print(f'Genes:   {N_GENES:,}')
print(f'Samples: {n_samples:,}')
print(f'Tamaño:  ~6.8 GB (sin comprimir)')
print(f'\nPrimeras 6 columnas: {all_col_names[:6]}')

peek.iloc[:, :6]

In [ ]:
# every 100th sample so we don't kill the laptop - ~196 samples is plenty for stats
selected_indices = [0, 1] + list(range(2, len(all_col_names), SAMPLE_STEP))
selected_cols    = [all_col_names[i] for i in selected_indices]

# clean up column names - spark doesn't like hyphens or dots in col names
clean_names = [c.replace('-', '_').replace('.', '_') for c in selected_cols]
sample_cols = clean_names[2:]  # everything after Name and Description

print(f'Usando {len(sample_cols)} columnas de muestra (cada {SAMPLE_STEP}a muestra de las {n_samples:,})')

# load with spark - filter to lines starting with ENSG to skip the 3 GCT header rows
# all real gene rows have ENSEMBL IDs starting with ENSG, clean and no shuffle needed
raw_rdd = spark.sparkContext.textFile(FILE_PATH)

data_rdd = (
    raw_rdd
    .filter(lambda line: line.startswith('ENSG'))   # skip version, dimensions, and header rows
    .map(lambda line: line.split('\t'))
    .map(lambda fields: [fields[i] for i in selected_indices])
)

df_raw = data_rdd.toDF(clean_names)
print(f'\nDataFrame cargado: {df_raw.count():,} genes x {len(clean_names)} columnas')

In [ ]:
# cast to float - spark reads text files as strings by default
df = df_raw
for c in sample_cols:
    df = df.withColumn(c, F.col(c).cast(FloatType()))

df.show(5, truncate=True)

## 2. Estadísticas Descriptivas

Se calculan estadísticas básicas sobre las columnas de expresión para entender la distribución de valores TPM. Los valores de TPM típicamente siguen una distribución **log-normal**: la mayoría de los genes tienen expresión baja o nula, con unos pocos genes altamente expresados formando una cola larga hacia la derecha.

In [ ]:
# quick describe on a few columns to get a feel for the data
df.select(["Name", "Description"] + sample_cols[:5]).describe().show()

In [ ]:
# compute stats for first 10 sample columns and put in a readable pandas table
stats_rows = []
for c in sample_cols[:10]:
    row = df.select(
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.avg(c).alias("mean"),
        F.stddev(c).alias("std")
    ).collect()[0]
    stats_rows.append({"muestra": c[:30], **row.asDict()})

pd.DataFrame(stats_rows).round(3)

## 3. Valores Faltantes y Ceros

El dataset GTEx no tiene valores `NULL` reales - cada gen recibe un valor TPM por muestra. Sin embargo, un valor `TPM = 0.0` puede significar dos cosas distintas:

- El gen genuinamente **no se expresa** en ese tejido
- La expresión está **por debajo del umbral de detección** del secuenciador

Esta ambigüedad es uno de los principales problemas de preprocesamiento del dataset y motiva el análisis de umbral en la siguiente sección.

In [ ]:
# check for actual nulls - should be basically zero in GTEx
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in sample_cols[:10]
]).toPandas().T.rename(columns={0: "nulls"})

# zeros are the real story - TPM=0 doesn't mean missing, it means not detected
zero_counts = df.select([
    F.count(F.when(F.col(c) == 0.0, c)).alias(c)
    for c in sample_cols[:10]
]).toPandas().T.rename(columns={0: "zeros"})

summary = null_counts.join(zero_counts)
summary["pct_zeros"] = (summary["zeros"] / N_GENES * 100).round(2)
print(summary.to_string())

## 4. Distribución de Expresión

Los valores de TPM tienen una distribución fuertemente asimétrica hacia la derecha - la mayoría de los genes tienen valores muy bajos o cero, con una cola de genes altamente expresados. En el histograma log1p esto se ve como dos picos (bimodal): un pico grande de genes con expresión baja/nula y un pico más pequeño de genes activamente expresados.

La media a través de todas las muestras seleccionadas da una visión global de cuánto se expresa cada gen en el conjunto de tejidos.

In [ ]:
# compute mean TPM per gene across all sampled columns
# use reduce to add Column objects - cleaner than sum() with generators
n_cols     = len(sample_cols)
total_expr = reduce(lambda a, b: a + b, [F.col(c) for c in sample_cols])

df_mean = df.withColumn("mean_tpm", total_expr / n_cols)

# pull to pandas - 59k rows and 3 cols, small enough
mean_pd = df_mean.select("Name", "Description", "mean_tpm").toPandas()
mean_pd["mean_tpm"] = pd.to_numeric(mean_pd["mean_tpm"], errors="coerce")

print(f"Genes analizados: {len(mean_pd):,}")
mean_pd[["mean_tpm"]].describe().round(4)

In [ ]:
# THRESHOLD is imported from GlobalVariables
total     = len(mean_pd)
expressed = int((mean_pd['mean_tpm'] > THRESHOLD).sum())
low_expr  = total - expressed

print(f'Umbral: mean TPM > {THRESHOLD}\n')
print(f"{'Genes totales':<40} {total:>7,}")
print(f"{'Genes expresados (> umbral)':<40} {expressed:>7,}  ({expressed/total*100:.1f}%)")
print(f"{'Genes filtrados (ruido/no expresados)':<40} {low_expr:>7,}  ({low_expr/total*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de Expresión Génica - GTEx V10', fontsize=14, fontweight='bold')

log_all = np.log1p(mean_pd['mean_tpm'].dropna())
log_thr = np.log1p(mean_pd[mean_pd['mean_tpm'] > THRESHOLD]['mean_tpm'].dropna())

# all genes - shows the bimodal distribution with noise peak on the left
axes[0].hist(log_all, bins=100, color='#4C72B0', alpha=0.85, edgecolor='none')
axes[0].axvline(np.log1p(THRESHOLD), color='#DD4949', lw=2, linestyle='--',
                label=f'Umbral = {THRESHOLD} TPM (log1p = {np.log1p(THRESHOLD):.2f})')
axes[0].set_xlabel('log1p(Mean TPM)', fontsize=11)
axes[0].set_ylabel('Numero de genes', fontsize=11)
axes[0].set_title(f'Todos los genes (n={total:,})')
axes[0].legend(fontsize=9)

# after threshold - cleaner, just the expressed genes
axes[1].hist(log_thr, bins=80, color='#DD8452', alpha=0.85, edgecolor='none')
axes[1].set_xlabel('log1p(Mean TPM)', fontsize=11)
axes[1].set_ylabel('Numero de genes', fontsize=11)
axes[1].set_title(f'Genes expresados - mean TPM > {THRESHOLD} (n={expressed:,})')

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'expression_threshold.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {out_path}')

In [ ]:
# top 20 most expressed genes - expect albumin (liver), hemoglobin (blood), etc.
top20 = mean_pd.nlargest(20, 'mean_tpm')[['Description', 'mean_tpm']].reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=top20, x='mean_tpm', y='Description', palette='viridis', orient='h')
plt.xlabel('Mean TPM (muestras de tejido seleccionadas)', fontsize=11)
plt.title('Top 20 Genes Mas Expresados - GTEx V10', fontsize=13, fontweight='bold')
plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'top_genes.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {out_path}')

print('\nTop 20 genes:')
print(top20.to_string(index=False))

## 5. Preprocesamiento: Problemas y Correcciones

Se identificaron los siguientes problemas en el dataset. Los problemas #1 y #2 se corrigen en este notebook. Los restantes requieren los metadatos de tejido del portal GTEx y se documentan para trabajo futuro.

| # | Estado | Problema | Descripción | Corrección |
|---|--------|----------|-------------|------------|
| 1 | Corregido | **Filas de metadatos GCT** | El formato GCT 1.2 tiene 2 filas antes del encabezado real - `spark.read.csv` las trataría como datos | Filtrar con `.filter(startswith('ENSG'))` en RDD - implementado en la celda de carga |
| 2 | Corregido | **IDs ENSEMBL no legibles** | Los genes tienen formato `ENSG00000000003.15`, no interpretable directamente | La columna `Description` del archivo GCT ya contiene el símbolo génico legible - celda siguiente |
| 3 | Pendiente | **Outliers extremos de TPM** | Algunos genes tienen TPM muy por encima de la media global | Requiere análisis IQR por tejido - pendiente de metadatos de tejido |
| 4 | Pendiente | **Muestras desbalanceadas por tejido** | No todos los 54 tejidos tienen el mismo número de donantes | Requiere cruzar IDs de muestra con `GTEx_v10_sample_attributes.txt` |
| 5 | Pendiente | **Ceros estructurales** | `TPM = 0` puede significar no expresado o por debajo del umbral de detección | Requiere análisis por tejido para distinguir los dos casos |
| 6 | Pendiente | **IDs de muestra opacos** | `GTEX-1117F-0005-SM-HL9SH` no contiene info de tejido directamente | Cruzar con `GTEx_v10_sample_attributes.txt` del portal GTEx |

In [ ]:
# fix #2 - Description column already has the human-readable gene symbol
# no external GTF mapping needed, it's right there in the file

print("ENSEMBL ID → Gene Symbol (primeros 10 genes):")
df.select("Name", "Description").show(10, truncate=False)

# build a lookup dict for downstream use
ensembl_to_symbol = dict(
    df.select("Name", "Description")
      .rdd.map(lambda r: (r["Name"], r["Description"]))
      .collect()
)
print(f"Lookup dict: {len(ensembl_to_symbol):,} entradas")

## Conclusiones

- El dataset GTEx V10 tiene **59,033 genes** × **19,614 muestras** en ~6.8 GB sin comprimir, lo que supera el requisito mínimo de 1 GB y justifica el uso de procesamiento distribuido con PySpark.

- El formato GCT 1.2 requiere manejo especial al cargar: las primeras 3 líneas son metadatos, no datos. Se resolvió filtrando líneas que comiencen con IDs ENSEMBL en el RDD.

- Los IDs ENSEMBL ya tienen su símbolo génico legible en la columna `Description` del mismo archivo - no se requiere tabla de mapeo externa.

- El dataset no contiene valores `NULL` - todas las celdas tienen un valor TPM. Sin embargo, una proporción alta de ceros por muestra indica que muchos genes no se detectaron en un tejido dado, lo cual es distinto a datos faltantes.

- Los valores TPM tienen una distribución log-normal asimétrica observable en el histograma: la mayoría de genes tienen TPM muy bajo y unos pocos tienen valores extremadamente altos.

## Referencias

1. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776

2. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex